<a href="https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My Lane: Lane 2 — Refresh / Content Opportunity Scoring**

I chose Lane 2 because I want to find which website pages a content team should review first. A company has limited time, so it needs to prioritize its pages. In this project, one row represents one page, and the final result will be a ranked list of pages. The people who would use this result are content editors or SEO leads. The repository already has some starter work, so I can use it as a reference. My choice is temporary, and I can change it before Week 4 ends. The data already shows some useful patterns, so I think this problem is worth exploring.

In [ ]:
# Setup: make sure we're running from the repo root, whether this is Colab or local (same pattern as notebooks/01).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faiqakashif82-netizen/FlyRank-MachineLearning-Internship"
REPO_DIR = "FlyRank-MachineLearning-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, 'data')) and here != os.path.dirname(here):
        here = os.path.dirname(here)
    os.chdir(here)

print('working directory:', os.getcwd())


working directory: /content/FlyRank-MachineLearning-Internship


In [ ]:
# Just showing the dataset is loadable and its basic shape before I reason about the lane.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('rows:', len(df))
print('columns:', len(df.columns))
print('unique clients:', df['client_id'].nunique())

rows: 30000
columns: 44
unique clients: 32


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The question:** Among a client's content pages that are already getting search traffic, which ones should a content editor review first for a refresh?

**Unit of analysis:** one content page (one row = one page, over its trailing 90-day window).

**The decision this improves:** which pages go to the top of a content editor's weekly review queue. Right now, without a ranked list, an editor either works alphabetically, by gut feeling, or waits for someone to flag a page — none of that uses the data that's already sitting there.

**Who acts, and what they do:** a content editor or SEO lead opens the top of the ranked queue and decides, page by page, whether to update the content, keep watching it, or leave it alone. The output is a list they can act on directly, not a black-box score.

**Cost of a wrong call:** two different mistakes, and they don't cost the same. If I rank a page as 'review me first' and it turns out fine, the cost is a wasted hour of an editor's time — annoying, but small. If I miss a page that is genuinely losing visibility and don't flag it, the cost is bigger: real traffic keeps declining unnoticed until someone finds it by accident. Because missing a real decline is more expensive than a false alarm, I care more about recall among the pages that truly need attention than about a perfect precision score.

**Why data or ML can help at all:** a simple rule (like 'flag every page older than 6 months') would catch far too many pages and overwhelm the editor. The signal that actually separates 'worth reviewing' from 'leave alone' is tangled across several things at once — traffic volume, how the trend is moving, position, freshness, engagement — and no single column tells the whole story on its own. That's exactly the kind of messy, many-signal pattern where a learned score can do better than an editor's gut or a single if-statement, while still staying explainable through reason codes.

In [ ]:
# No computation needed for this section — just confirming the columns I'm referring to above actually exist.
cols_i_mean = ['content_id', 'client_id', 'trend_direction', 'impressions_90d', 'avg_position',
               'days_since_last_update', 'engagement_rate', 'scroll_rate']
print([c for c in cols_i_mean if c in df.columns])

['content_id', 'client_id', 'trend_direction', 'impressions_90d', 'avg_position', 'days_since_last_update', 'engagement_rate', 'scroll_rate']


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

I loaded `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients) and checked a few numbers that speak directly to this lane: is there a real mix of pages worth ranking, and does the data separate them at all?

In [ ]:
# Following the starter's own qualifying rule: enough exposure to matter, and old enough to have a real trend.
qualified = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]
print('qualified rows:', len(qualified))

pct_declining = (qualified['trend_direction'] == 'down').mean() * 100
print(f'share of qualified pages trending down: {pct_declining:.1f}%')

declining_with_demand = qualified[(qualified['trend_direction'] == 'down') & (qualified['impressions_90d'] >= 100)]
print(f'declining pages that still have real demand (>=100 impressions/90d): {len(declining_with_demand)} '
      f'({len(declining_with_demand)/len(qualified)*100:.1f}% of qualified pages)')

low_ctr_visible = qualified[(qualified['impressions_90d'] >= 500) & (qualified['avg_position'] > 0) &
                              (qualified['avg_position'] <= 20) & (qualified['ctr'] < 0.5)]
print(f'visible pages (position 1-20, 500+ impressions) with weak CTR (<0.5%): {len(low_ctr_visible)} '
      f'({len(low_ctr_visible)/len(qualified)*100:.1f}% of qualified pages)')


qualified rows: 30000
share of qualified pages trending down: 54.2%
declining pages that still have real demand (>=100 impressions/90d): 13152 (43.8% of qualified pages)
visible pages (position 1-20, 500+ impressions) with weak CTR (<0.5%): 9759 (32.5% of qualified pages)


**What these numbers tell me:**

- **54.2%** of qualified pages are currently trending down — the pattern is common enough to be worth ranking, not a rare edge case.
- **43.8%** of qualified pages (13,152 pages) are declining *while still pulling in real impressions* — these are the pages where a wrong call is expensive, since they matter and are slipping at the same time.
- **32.5%** of qualified pages (9,759 pages) sit in the top 20 search positions with weak click-through — a different kind of opportunity than a raw decline, and a sign that one score alone won't capture everything worth flagging.

None of these numbers are tiny or huge — they sit in a middle range where ranking genuinely matters, because an editor clearly can't review 13,000+ pages by hand, but a plain age-based or alphabetical list would waste most of their time on pages that don't need it.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**

**Observed patterns:** which measured signals (traffic, position, freshness, engagement) tend to appear together on pages that are currently declining or under-performing.

**Directional signals:** whether a page's trend and context suggest it's more or less likely to need review than another page, based on what has already been measured.

**Decision-support output:** a ranked queue with reason codes, meant to help a human reviewer prioritize — never a guarantee about any single page.

**What I cannot and will not claim:**

I cannot claim that refreshing a page causes it to recover. Proving that would need an actual experiment (for example, refreshing some pages and holding others back to compare), which this data doesn't give me.

I cannot claim I've discovered or predicted anything about Google's ranking algorithm. I only have observed search and engagement numbers, not the algorithm itself.

I won't treat trend_direction or trend_pct as inputs to any model that predicts decline, since the label itself is built from those same columns — that would just be the model learning to copy a definition, not finding real signal.
I won't present a high score as certainty. Every ranked page still needs a human to look at it before any action is taken.

In [ ]:
# Nothing to compute here — this section is a written commitment, not a calculation.
print('Section 4 is a written commitment about claim language; no code needed.')

Section 4 is a written commitment about claim language; no code needed.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.